Multivariate Autoregression of algal Bloom Intensity using ocean-atmospheric variables

In [1]:
#load libraries
import ee
import geemap

In [2]:
# 1. Initialize Earth Engine
ee.Authenticate()
ee.Initialize()

Map = geemap.Map(center=[10, 80], zoom=5)

In [3]:
# 2. Study Area
northernio = ee.FeatureCollection("projects/ee-punyachinjuu/assets/NIO")
region = northernio.geometry()

In [4]:
# 3. Date Range
START_YEAR = 2003
END_YEAR   = 2020

years  = ee.List.sequence(START_YEAR, END_YEAR)
months = ee.List.sequence(1, 12)
TIME_FIELD = "system:time_start"

In [5]:
# 4. Helper Functions

# Clipping an imagecollection
def clip_collection(collection, band):
    return (ee.ImageCollection(collection)
            .select(band)
            .filterDate(f"{START_YEAR}-01-01", f"{END_YEAR}-12-31")
            .map(lambda img: img.clip(region)))
    
# Making monthly composites
def monthly_composite(collection, reducer='mean'):
    def make_month(y):
        def make_image(m):
            img = (collection
                   .filter(ee.Filter.calendarRange(y, y, 'year'))
                   .filter(ee.Filter.calendarRange(m, m, 'month')))

            img = img.mean() if reducer == 'mean' else img.max()

            return (img
                    .set('year', y)
                    .set('month', m)
                    .set(TIME_FIELD,
                         ee.Date.fromYMD(y, m, 1).millis()))
        return months.map(make_image)

    return ee.ImageCollection.fromImages(years.map(make_month).flatten())

# Add a time band 
def add_time_band(image):
    date = ee.Date(image.get(TIME_FIELD))
    t = date.difference(ee.Date(f"{START_YEAR}-01-01"), 'year')
    return (image
            .addBands(ee.Image.constant(1).rename('constant'))
            .addBands(ee.Image(t).rename('t').float()))

# Detrending
def detrend(collection, dependent):
    independents = ee.List(['constant', 't'])

    trend = (collection
             .select(independents.add(dependent))
             .reduce(ee.Reducer.linearRegression(2, 1)))

    coefficients = (trend.select('coefficients')
                    .arrayProject([0])
                    .arrayFlatten([independents]))

    def remove_trend(image):
        fitted = image.select(independents)\
                      .multiply(coefficients)\
                      .reduce('sum')
        return (image.select(dependent)
                .subtract(fitted)
                .rename(dependent)
                .copyProperties(image, [TIME_FIELD]))

    return collection.map(remove_trend)

# Creating lagged imagecollection
def lag_join(primary, secondary, lag_days):
    time_filter = ee.Filter.And(
        ee.Filter.maxDifference(
            difference=lag_days * 24 * 60 * 60 * 1000,
            leftField=TIME_FIELD,
            rightField=TIME_FIELD),
        ee.Filter.greaterThan(
            leftField=TIME_FIELD,
            rightField=TIME_FIELD)
    )

    return ee.Join.saveAll(
        matchesKey='images',
        ordering=TIME_FIELD,
        ascending=False
    ).apply(primary, secondary, time_filter)

# Merge lagged imagecollection
def merge_lagged(image):
    def merger(current, previous):
        return ee.Image(previous).addBands(current)

    return ee.ImageCollection.fromImages(
        image.get('images')
    ).iterate(merger, image)


In [6]:
# 5. Load Datasets

SST  = clip_collection('NASA/OCEANDATA/MODIS-Aqua/L3SMI', 'sst')
CHL  = clip_collection('NASA/OCEANDATA/MODIS-Aqua/L3SMI', 'chlor_a')
POC  = clip_collection('NASA/OCEANDATA/MODIS-Aqua/L3SMI', 'poc')
RF   = clip_collection('NASA/GPM_L3/IMERG_MONTHLY_V06', 'precipitation')
SSS  = clip_collection('HYCOM/sea_temp_salinity', 'salinity_0')
SSH  = clip_collection('HYCOM/sea_surface_elevation', 'surface_elevation')
WIND = clip_collection('NOAA/CDR/SST_PATHFINDER/V53', 'wind_speed')

In [7]:
# 6. Monthly Composites

monthly_CHL  = monthly_composite(CHL, reducer='max')
monthly_SST  = monthly_composite(SST)
monthly_POC  = monthly_composite(POC)
monthly_RF   = monthly_composite(RF)
monthly_SSS  = monthly_composite(SSS)
monthly_SSH  = monthly_composite(SSH)
monthly_WIND = monthly_composite(WIND)

In [8]:
# 7. Bloom Detection (>3.4 mg/m3)

def bloom_mask(image):
    mask = image.select('chlor_a').gt(3.4).selfMask()
    return image.addBands(mask.rename('Mask'))

monthly_CHL = monthly_CHL.map(bloom_mask)

In [9]:
# 8. Add Time Bands

chl_t  = monthly_CHL.map(add_time_band)
sst_t  = monthly_SST.map(add_time_band)
poc_t  = monthly_POC.map(add_time_band)
rf_t   = monthly_RF.map(add_time_band)
sss_t  = monthly_SSS.map(add_time_band)
ssh_t  = monthly_SSH.map(add_time_band)
wind_t = monthly_WIND.map(add_time_band)

In [10]:
# 9. Detrend All Variables

chl_dt  = detrend(chl_t,  'chlor_a')
sst_dt  = detrend(sst_t,  'sst')
poc_dt  = detrend(poc_t,  'poc')
rf_dt   = detrend(rf_t,   'precipitation')
sss_dt  = detrend(sss_t,  'salinity_0')
ssh_dt  = detrend(ssh_t,  'surface_elevation')
wind_dt = detrend(wind_t, 'wind_speed')

In [11]:
# 10. Lag Example (Bloom vs SST)

lagged = lag_join(chl_dt, sst_dt, lag_days=30)
merged = ee.ImageCollection(lagged.map(merge_lagged))

cov = (merged
       .select(['chlor_a', 'sst'])
       .map(lambda img: img.toArray())
       .reduce(ee.Reducer.covariance()))

def correlation(array_img):
    cov = array_img.arrayGet([0, 1])
    sd0 = array_img.arrayGet([0, 0]).sqrt()
    sd1 = array_img.arrayGet([1, 1]).sqrt()
    return cov.divide(sd0).divide(sd1).rename('correlation')

corr_map = correlation(cov)

In [12]:
# 11. Multivariate Autoregression

combined = (chl_dt
            .combine(sst_dt)
            .combine(sss_dt)
            .combine(ssh_dt)
            .combine(poc_dt)
            .combine(rf_dt)
            .combine(wind_dt)
            .map(add_time_band))

independents = ee.List([
    'constant', 'sst', 'salinity_0',
    'surface_elevation', 'poc',
    'precipitation', 'wind_speed'
])

dependent = 'chlor_a'

ar_model = (combined
            .select(independents.add(dependent))
            .reduce(ee.Reducer.linearRegression(
                independents.length(), 1)))

coefficients = (ar_model.select('coefficients')
                .arrayProject([0])
                .arrayFlatten([independents]))

In [13]:
# Optional smoothing
smoothed = coefficients.reduceNeighborhood(
    reducer=ee.Reducer.mean(),
    kernel=ee.Kernel.circle(15)
)

In [14]:
# 12. Export

geemap.ee_export_image_to_drive(
    smoothed,
    description='Autoreg_Coefficients',
    folder='REGRESSION',
    region=region,
    scale=4000
)